# Tracing a data citation: a guide to the SRAD tables

This notebook answers one question, several ways: **why was this data citation classified the
way it was?**

The public release tells you *that* an article cites a dataset. It also tells you whether we
judged the use Primary (the data was generated for that article) or Secondary (the data was
reused). SRAD tells you **why**. It gives you enough detail to check our reasoning, and to
disagree with us.

Figures quoted below refer to release 202609 (rebuilt 26 September 2026). The next release,
202610, is due on 2 October 2026.

## The mental model

Classification works as a cascade of tiers. Cheap, strong evidence is tried first. Each later
tier only sees the citations the earlier tiers left undecided. So every citation was decided by
exactly one tier, and each tier used its own kind of evidence.

That is why the tables are laid out like this:

```
srad.citations          the spine - one row per (release, article, dataset). Read `tier`.
   |
   +-- tier tells you which ONE evidence table to join:
   |      repository_evidence -> srad.evidence_repository
   |      metadata_similarity -> srad.evidence_metadata
   |      mention_cue         -> srad.evidence_cues
   |      model               -> srad.evidence_model
   |      llm                 -> srad.evidence_llm
   |      registered_relation -> (no evidence table: DataCite asserts it directly)
   |      family_prior        -> (no evidence table: see srad.accession_patterns)
   |      none                -> undecided; nothing was sufficient
   |
   +-- srad.mentions        where the identifier appears, and the sentence (open access only)
   +-- srad.rules           what each rule actually is, and how well it measures
   +-- srad.methods         the classifier and the LLM, including the full prompt
   +-- srad.releases        per-release counts and benchmark scores
```

You never need to join all the evidence tables. Read the tier, then join the one table it names.

## Setup

You will need `google-cloud-bigquery` and `pandas`, plus a Google account with access to the
SRAD dataset.

Every query is filtered on `release`. Pinning a release is what makes your answer reproducible:
the same query on the same release always gives the same result.

The first query lists the available releases with their headline figures. `srad.releases` holds
one row per release. Its columns are:

- `citations`, `articles`, `datasets` — the size of the release.
- `pct_classified` — the share of citations labelled Primary or Secondary (92.15% in 202609).
- `pct_open_access` — the share of citations in open-access articles.
- `agreement_kaggle`, `coverage_kaggle` — scores against the Make Data Count Kaggle competition labels.
- `agreement_rdmpage`, `coverage_rdmpage` — scores against the rdmpage labels.
- `agreement_combined`, `coverage_combined` — scores against both gold sets together. This is the headline benchmark.
- `gold_pairs_combined` — how many (article, dataset) pairs the combined score is based on.
- `gold_pairs_discordant` — pairs the two gold sets label differently. These are dropped, because no pipeline can agree with both.
- `pipeline_commit`, `notes` — which version of the code produced the release.

*Agreement* is the share of classified gold pairs where our label matches the gold label.
*Coverage* is the share of gold pairs we classify at all. For 202609 the combined figures are
0.9617 agreement and 0.9078 coverage, over 1,063 pairs. The query below shows only some of these
columns; add the others if you want them.

In [ ]:
from google.cloud import bigquery
import pandas as pd

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

PROJECT = "ds-make-data-count"   # billing project for your queries
SRAD    = "ds-make-data-count.srad"
RELEASE = "202609"               # pin a release; see srad.releases for what is available

client = bigquery.Client(project=PROJECT)

def q(sql, **params):
    """Run a query and return a DataFrame. `RELEASE` and `SRAD` are substituted for you."""
    return client.query(sql.format(SRAD=SRAD, RELEASE=RELEASE, **params)).to_dataframe()

q("SELECT release, citations, articles, pct_classified, agreement_kaggle, agreement_rdmpage "
  "FROM `{SRAD}.releases` ORDER BY release DESC")

## 1. Which tier decided what

Start here. It shows where the decisions actually come from, and the answer surprises most
people: the machine-learning tiers are the small ones.

In [ ]:
q("""
SELECT tier, COUNT(*) AS citations,
       ROUND(100*COUNT(*)/SUM(COUNT(*)) OVER (), 1) AS pct,
       COUNT(DISTINCT rule_applied) AS n_rules,
       STRING_AGG(DISTINCT type ORDER BY type) AS decisions
FROM `{SRAD}.citations`
WHERE release = '{RELEASE}'
GROUP BY tier ORDER BY citations DESC
""")

Repository evidence decides about half of all citations (51% in 202609). This is the deposit record itself: who submitted
the data, when, and which publication the archive links it to. It is also the most precise
evidence the pipeline has. The LLM tier, by contrast, decides about 0.1%.

That order is deliberate. Whether data was generated for an article is really a *fact about the
deposit*, and the archive holds that fact. We only fall back to reading sentences when no such
fact is available.

Look at the `none` tier too. In 202609 about one citation in thirteen (686,721 of 8,749,547) is
shipped as explicitly **Unclassified** rather than guessed. Please do not filter these rows out
silently. Section 9 explains why.

## 2. Trace one article, end to end

This is the query most people want. Give it a DOI, and it returns every dataset that article
cites, together with:

- the decision (Primary, Secondary or Unclassified);
- the rule that made it, in plain words;
- that rule's measured precision, where we have one;
- the sentence where the identifier appears (open-access articles only).

In [ ]:
q("""
DECLARE target_doi STRING DEFAULT '10.1186/s12985-024-02479-8';

SELECT c.dataset_id, c.id_kind, c.family_or_repo, c.type, c.rule_applied, c.tier,
       r.definition        AS why,
       r.measured_precision AS rule_precision,
       m.location, m.sentence_is_withheld,
       SUBSTR(m.sentence, 1, 130) AS sentence
FROM `{SRAD}.citations` c
LEFT JOIN `{SRAD}.rules` r
  ON r.rule_id = c.rule_applied
LEFT JOIN `{SRAD}.mentions` m
  ON  m.publication_id = c.publication_id AND m.dataset_id = c.dataset_id
  AND m.release = c.release AND m.mention_ordinal = 1
WHERE c.release = '{RELEASE}' AND LOWER(c.article_doi) = LOWER(target_doi)
ORDER BY c.dataset_id
""")

## 3. Follow the evidence for one tier

The spine tells you *which* rule fired. The evidence table tells you *what that rule saw*. For a
`repository_evidence` decision, that means the deposit record, so you can check our reasoning
against it directly.

The columns in `srad.evidence_repository` are:

- `family` — the repository family, for example `gen` for GenBank/ENA/DDBJ.
- `decision` — what the deposit evidence concluded (see the list below the query).
- `record_found` — whether we hold a deposit record for this accession at all.
- `submitters` — the names on the deposit record.
- `linked_pubmed`, `linked_doi` — the publication(s) the archive links the record to.
- `dep_year` — the year the data was deposited.
- `submitter_overlap`, `center_name` — author overlap and submitting centre, for non-GenBank families.
- `submit_affiliation` — the submitters' affiliation, for GenBank records.
- `reference_authors` — for GenBank, every author in the record's reference block, not just the submitters. The protein companion-paper check (below) uses this column.
- `evidence_table` — which source the evidence came from. See the note below.

**Where GenBank evidence comes from.** GenBank is by far the largest family (about 4.05 million
citations in 202609). Since September we fetch attribution directly from NCBI, for the
nucleotide and protein records that are actually cited. An older mirror of GenBank flat files is
kept, frozen, as a fallback. So for `gen` rows, `evidence_table` reads:

- `insdc_attribution` — from the targeted NCBI fetch;
- `insdc_provenance` — from the frozen mirror;
- `insdc_attribution + insdc_provenance` — from both;
- `none` — we found no record in either source.

For other families it reads `repo_metadata / sra_bq_metadata`.

In [ ]:
q("""
DECLARE target_doi STRING DEFAULT '10.1186/s12985-024-02479-8';

SELECT c.dataset_id, c.rule_applied,
       e.decision AS deposit_decision,
       e.submitters, e.linked_pubmed, e.dep_year, e.record_found, e.evidence_table
FROM `{SRAD}.citations` c
JOIN `{SRAD}.evidence_repository` e
  ON e.publication_id = c.publication_id AND e.dataset_id = c.dataset_id
 AND e.release = c.release
WHERE c.release = '{RELEASE}' AND LOWER(c.article_doi) = LOWER(target_doi)
""")

Three values in `decision` need care. They mean *we had no usable evidence*, not *the answer was
no*:

- **`no_metadata_yet`** — we hold no deposit record for this accession.
- **`undecided`** — we hold a record, but it carried neither a linked publication nor submitter
  names. There was nothing to compare against the article.
- **`held_protein_companion`** — a GenBank protein record, held back on purpose. Protein records
  from genome projects often list a sequencing centre as submitter and the genome paper's large
  author list in their references. If the citing article shares an author with that reference
  list, and appeared within a year of the deposit, it may be a companion paper from the same
  project. Calling that "reuse" could be wrong, so we leave it unclassified instead (520
  citations in 202609). You can check the reasoning yourself by comparing `reference_authors`
  with the article's authors.

Reading any of these as evidence of reuse is the most common way to misread this dataset. Absence
of evidence is not evidence of absence.

For the DOI arm, join `evidence_metadata` instead. It carries the dataset's creator names and the
article's author names. Those lists matter. If `author_overlap_frac` is 0, the names show you
why: a genuinely different research group, the same people written in a different format, or an
organisation listed as depositor.

## 4. Inspect the rules

Every rule is published with its definition and, where we have measured it, its precision.

`definition_kind` tells you what sort of definition it is. Some rules are regular expressions.
Some are threshold tests on similarity features. One is a probability cut-off, and one is an LLM
verdict.

In [ ]:
q("""
SELECT r.rule_id, r.tier, r.decision, r.definition_kind,
       r.measured_precision, r.n_labels,
       COUNT(c.rule_applied) AS citations_decided
FROM `{SRAD}.rules` r
LEFT JOIN `{SRAD}.citations` c
  ON c.rule_applied = r.rule_id AND c.release = '{RELEASE}'
GROUP BY 1,2,3,4,5,6
ORDER BY citations_decided DESC
LIMIT 12
""")

**`measured_precision` is NULL for most rules. That is deliberate, not missing data.**

We fill it in only where the rule has been measured against labels drawn from the citations it
actually decides. Borrowing a precision figure from a different population is an easy mistake,
and a badly misleading one. It led to two wrong conclusions during development. So where we have
not measured, we publish nothing rather than something reassuring.

Before you rely on a rule, read its `notes`. Several record known failure modes.

In [ ]:
q("""
SELECT rule_id, definition, notes
FROM `{SRAD}.rules`
WHERE rule_id IN ('llm_primary','metadata_match','repo_linked_other_secondary','model_primary')
""")

## 5. The accession parsers

Accession identifiers are found using a regular expression for each repository. For every family
we publish the pattern, its false-positive risk, and whether it is excluded from regex discovery.
That way you can see exactly what could, and could not, have been found.

In [ ]:
q("""
SELECT family, pattern, fp_risk, default_type_prior, excluded_from_regex_discovery
FROM `{SRAD}.accession_patterns`
WHERE family IN ('gen','pdb','geo','biosample','uniprot')
ORDER BY family
""")

One caution applies to any pattern-based extraction, including ours: **a pattern that is too
narrow fails silently.** An accession that does not match looks exactly like one that was never
mined. So read format-validity rates as a floor, not as a measure of extraction quality.

Families with `fp_risk = 'high'` are excluded from regex *discovery*. We accept them when a
trusted source mined them, but we do not search for them ourselves, because their patterns clash
with grant numbers, specimen codes and similar strings.

A pattern that is too broad causes the opposite problem. Some strings are valid PDB IDs yet are
something else entirely. Two `srad.rules` entries, `band_notation` and `glycan_notation`, exclude
them. `band_notation` catches chromosome bands such as "7q22.1" or "19q13.2". `glycan_notation`
catches sugar linkages such as "Galα1-3Gal". Only the surrounding text can tell these apart from
real structures. Together the two rules removed about 12,100 false PDB citations in September.

## 6. Sentences, locations, and what is withheld

**Location is available for every mention. Sentence text is available only for open-access
articles.** This is a licensing constraint, not a choice.

In [ ]:
q("""
SELECT is_open_access,
       COUNT(*) AS mentions,
       COUNTIF(sentence IS NOT NULL) AS with_sentence,
       COUNTIF(sentence_is_withheld) AS withheld,
       COUNT(DISTINCT location) AS distinct_locations
FROM `{SRAD}.mentions`
WHERE release = '{RELEASE}'
GROUP BY is_open_access ORDER BY is_open_access DESC
""")

`sentence_is_withheld` lets you tell two kinds of NULL apart:

- a missing sentence because the article is closed access is a *licensing* fact;
- a missing sentence because no mention was found is an *evidence* fact.

They support opposite conclusions, and without the flag they look identical.

`location` is the section of the article where the identifier appears, mapped to the GROBID
schema: `availability_sentences`, `body_sentences`, `references_titles`, `tables_rows`, and so
on. It is often as informative as the sentence. A DOI in a data-availability statement means
something different from the same DOI in a reference list.

For the DOI arm, `original_location` records where the identifier was physically printed, before
we traced a reference entry back to the sentence that cites it.

## 7. The methods that read sentences

Two components classify by reading rather than by rule:

- a gradient-boosted classifier over structural features;
- a language model that reads the mention sentence.

Both are published with their provenance. For the LLM we also publish the **full prompt**. With a
zero-shot model, the prompt *is* the method.

In [ ]:
m = q("SELECT method_id, kind, name, version, endpoint, training_population, "
      "precision_by_class, applies_to_families FROM `{SRAD}.methods`")
display(m)

print(q("SELECT prompt_text FROM `{SRAD}.methods` WHERE method_id = 'llm_doi_primary_v3'")
        .iloc[0]["prompt_text"])

Look at `precision_by_class` for the LLM. Its PRIMARY verdict measures 0.937 and is used. Its
SECONDARY verdict measures 0.560 and is **never** used.

Why the difference? Explicit wording such as "the data underpinning the analysis reported in
this paper" is real evidence that the data was generated for the paper. But the *absence* of such
wording is not evidence of reuse. It is just absence. So we keep one verdict and discard the other.

The classifier shows the same asymmetry in `applies_to_families`. Its Secondary verdicts are
allowed everywhere. Its Primary verdicts are allowed only in repository families where it has
training labels.

## 8. A worked analysis: reuse rates by repository

Once the decisions and their provenance sit in one place, ordinary questions become ordinary
queries. Here we compare Primary rates across repositories.

In [ ]:
df = q("""
SELECT family_or_repo AS repository,
       COUNT(*) AS citations,
       ROUND(100*COUNTIF(type='Primary')
             /NULLIF(COUNTIF(type IN ('Primary','Secondary')),0), 1) AS pct_primary,
       ROUND(100*COUNTIF(type='Unclassified')/COUNT(*), 1) AS pct_unclassified
FROM `{SRAD}.citations`
WHERE release = '{RELEASE}' AND id_kind = 'accession'
GROUP BY 1 HAVING citations > 10000
ORDER BY citations DESC LIMIT 12
""")
df

Read that table with the tiers in mind. It shows clearly why provenance matters.

UniProt, RefSeq, Ensembl, Pfam and Cellosaurus sit at 0% Primary. That is not a finding about
research practice. It comes from the `reference_db_prior` rule, which labels citations of
reference databases as Secondary by default, on the reasoning that nobody deposits their own data
*into* UniProt. Look the rule up in `srad.rules` and you will see it has no measured precision.

Now compare BioProject with PXD (PRIDE). Both are close to 58% Primary in 202609, but for different
reasons: BioProject is decided mostly from deposit records (78% of its citations), PXD mostly
from cue words around the mention (64%), and PXD leaves far more unclassified (30%, against 13%). The next query shows the split by tier. That is three very different kinds of
evidence in a single column.

**Always group by `tier` before drawing a conclusion from `type`.**

In [ ]:
q("""
SELECT family_or_repo AS repository, tier, COUNT(*) AS citations
FROM `{SRAD}.citations`
WHERE release = '{RELEASE}' AND family_or_repo IN ('uniprot','bioproject','pxd','gen')
GROUP BY 1,2 ORDER BY repository, citations DESC
""")

## 9. What this dataset will not tell you

Please read this before you build on the data.

In [ ]:
q("""
SELECT
  (SELECT COUNT(*) FROM `{SRAD}.rules` WHERE measured_precision IS NULL)
     AS rules_never_measured,
  (SELECT COUNT(*) FROM `{SRAD}.rules`) AS rules_total,
  (SELECT COUNTIF(type = 'Unclassified') FROM `{SRAD}.citations`
     WHERE release = '{RELEASE}') AS unclassified_citations,
  (SELECT COUNTIF(sentence_is_withheld) FROM `{SRAD}.mentions`
     WHERE release = '{RELEASE}') AS sentences_withheld
""")

- **Most rules have no measured precision.** We publish NULL rather than an estimate.
- **About one citation in thirteen is Unclassified** (686,721 in 202609). Dropping those rows
  biases any rate you compute, because they are not spread evenly. Most sit in two places: GenBank
  records that carry neither submitter names nor a publication link (many are older EST, GSS and
  patent records), and DOI-arm citations found only in reference lists. Report them, or say
  clearly that you excluded them.
- **Sentences are withheld for closed-access articles.** Locations are not.
- **We do not record which rule was *overruled*.** Later tiers overwrite earlier decisions, so you
  can see which rule decided but not what it replaced. This is a known limitation.
- **Extraction depends on the full text we hold.** An article we do not hold contributes nothing,
  and that gap is not visible in this data. Use `srad.releases` and the recall audit in the
  methods documentation for a sense of scale.
- **The benchmark barely covers GenBank proteins.** The gold sets contain almost no GenBank
  protein citations and few nucleotide ones. So the roughly 260,000 GenBank citations newly
  classified in September rest on rule-level evidence in `srad.rules`, not on the benchmark.
- **`evidence_llm.method_id_provenance`** says `asserted_from_generating_script` for early
  releases. There, the endpoint was recorded in the generating code rather than next to each
  verdict. We can tell you which model was used, but the data does not prove it by itself.

## Reproducibility

Everything above is filtered on one `release`. Tables are partitioned by release, so a pinned
release keeps giving the same answer, and two releases can be compared directly.

`srad.releases` records the counts and the benchmark agreement and coverage measured when the
release was built. When you cite a release identifier, those figures come with it.

We welcome corrections and disagreements. Many of the adjudicated cases behind our own precision
figures started out exactly that way.